# Large Language Model (Generative text transformer) 

A generative use-case of transformers is text generation. The model is trained to predict the next word in a sequence, given the previous words. This is done by feeding the model a sequence of words and asking it to predict the next word. The model can then be used to generate new text by repeatedly predicting the next word and feeding it back into the model.

Changes (relative to the binary classifier with a transformer backbone):
- output head: instead of outputting a class, it outputs a word from the vocabulary (hidden_d --> vocab_size)
- prediction: instead of using the <CLS> token, we use all the tokens to predict the next word
- attention mask: we need to add an attention mask to prevent the model from attending to future tokens (causal attention)  

In [27]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets.mnist import MNIST
from torchvision.transforms import ToTensor
from tqdm import tqdm, trange

In [28]:
class SimpleTokenizer:
    def __init__(self, texts):
        vocab = set()
        for t in texts:
            vocab.update(t.lower().split())
        self.word2idx = {w: i+2 for i, w in enumerate(vocab)}
        self.word2idx["<PAD>"] = 0
        self.word2idx["<CLS>"] = 1
        self.idx2word = {i: w for w, i in self.word2idx.items()}

    def encode(self, text, max_len):
        tokens = ["<CLS>"] + text.lower().split()
        ids = [self.word2idx.get(t, 0) for t in tokens]
        ids = ids[:max_len]
        ids += [0] * (max_len - len(ids))
        return torch.tensor(ids)

In [29]:
def get_positional_embeddings(seq_len, d):
    pos = torch.zeros(seq_len, d)
    for i in range(seq_len):
        for j in range(d):
            pos[i, j] = (
                np.sin(i / (10000 ** (j / d)))
                if j % 2 == 0
                else np.cos(i / (10000 ** ((j - 1) / d)))
            )
    return pos

Add causal attention mask:
- token 0 can attend to: [0]
- token 1 can attend to: [0, 1]
- token 2 can attend to: [0, 1, 2]

It must not attend to future tokens. This is enforced by an upper‑triangular mask:

[0  -∞  -∞]

[0   0  -∞]

[0   0   0]

After softmax, all -∞ positions become zero probability.

In [30]:
# Multi-headed Self Attention - causal attention mask
class MyMSA(nn.Module):
    def __init__(self, d, n_heads=2):
        super(MyMSA, self).__init__()
        self.d = d
        self.n_heads = n_heads

        assert d % n_heads == 0, f"Can't divide dimension {d} into {n_heads} heads"

        d_head = int(d / n_heads)
        self.q_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.k_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.v_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.d_head = d_head
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequences):
        # Sequences has shape (N, seq_length, token_dim)
        # We go into shape    (N, seq_length, n_heads, token_dim / n_heads)
        # And come back to    (N, seq_length, item_dim)  (through concatenation)
        result = []
        for sequence in sequences:
            seq_len, _ = sequence.shape
            seq_result = []
            for head in range(self.n_heads):
                q_mapping = self.q_mappings[head]
                k_mapping = self.k_mappings[head]
                v_mapping = self.v_mappings[head]

                seq = sequence[:, head * self.d_head : (head + 1) * self.d_head]
                q, k, v = q_mapping(seq), k_mapping(seq), v_mapping(seq)
                scores = q @ k.T / (self.d_head**0.5)

                # causal attention mask
                # 0  -∞  -∞
                # 0   0  -∞
                # 0   0   0
                mask = torch.triu(
                    torch.ones(seq_len, seq_len, device=scores.device),
                    diagonal=1
                ).bool()

                scores = scores.masked_fill(mask, float("-inf"))


                attention = self.softmax(scores)
                seq_result.append(attention @ v)
            result.append(torch.hstack(seq_result))
        return torch.stack(result)


In [31]:


class MyTransformerBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyTransformerBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MyMSA(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d),
        )

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        return out

Change the output head to predict the next word:

In [32]:
class MyTextTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        hidden_d=64,
        n_heads=4,
        n_blocks=2,
        n_classes=2,
    ):
        super().__init__()

        self.hidden_d = hidden_d
        self.max_len = max_len

        # Token embedding
        self.embedding = nn.Embedding(vocab_size, hidden_d)

        # Positional embedding
        self.register_buffer(
            "pos_embedding",
            get_positional_embeddings(max_len, hidden_d),
            persistent=False,
        )

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [MyTransformerBlock(hidden_d, n_heads) for _ in range(n_blocks)]
        )

        # next word generation head
        self.lm_head = nn.Linear(hidden_d, vocab_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        batch_size, seq_len = x.shape
        # Embeddings
        emb = self.embedding(x)
        emb = emb + self.pos_embedding[:seq_len].unsqueeze(0)

        # Transformer blocks
        for block in self.blocks:
            emb = block(emb)
        
        # Project to vocabulary
        logits = self.lm_head(emb)

        return logits


In [37]:
np.random.seed(0)
torch.manual_seed(0)

# Loading data
texts = [
    "I like bikes",
    "I like cookies",
    "you like cookies",
    "you like bikes",
    "she likes bikes",
    "he likes bikes",
    "we like bikes"
]

# Defining model and training options
tokenizer = SimpleTokenizer(texts)
max_len = 6

X = torch.stack([tokenizer.encode(t, max_len) for t in texts])

model = MyTextTransformer(
    vocab_size=len(tokenizer.word2idx),
    max_len=max_len,
    hidden_d=32,
    n_heads=4,
    n_blocks=2,
)

# Training loop
N_EPOCHS = 200
LR = 0.005
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in trange(N_EPOCHS, desc="Training"):
    model.train()
    optimizer.zero_grad()
    
    # 🔹 shift inputs and targets
    X_in = X[:, :-1]     # input tokens
    X_target = X[:, 1:]  # next-token targets

    # 🔹 forward pass
    logits = model(X_in)  # (B, T-1, vocab_size)

    # 🔹 reshape for CrossEntropy
    loss = criterion(
        logits.reshape(-1, logits.size(-1)),
        X_target.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss {loss.item():.4f}")


Training:   2%|▏         | 3/200 [00:00<00:18, 10.90it/s]

Epoch 0 | Loss 2.7043


Training:  12%|█▏        | 24/200 [00:01<00:08, 21.21it/s]

Epoch 20 | Loss 0.4119


Training:  22%|██▎       | 45/200 [00:02<00:07, 20.24it/s]

...

Training:  81%|████████  | 162/200 [00:08<00:01, 20.95it/s]

Epoch 160 | Loss 0.3892


Training:  92%|█████████▏| 183/200 [00:09<00:00, 21.15it/s]

Epoch 180 | Loss 0.3892


Training: 100%|██████████| 200/200 [00:10<00:00, 19.43it/s]


In [39]:
# testing the trained model
test_texts = [
    "I like",
    "you",
    "likes"   
]
X_test = torch.stack([tokenizer.encode(t, max_len) for t in test_texts])

def generate_next_tokens(model, tokenizer, texts, max_len):
    model.eval()

    X = torch.stack([tokenizer.encode(t, max_len) for t in texts])

    with torch.no_grad():
        for pos in range(X.size(1) - 1):
            logits = model(X[:, :pos + 1])          # (B, pos+1, V)
            next_token = torch.argmax(
                logits[:, -1], dim=-1
            )                                       # (B,)
            X[:, pos + 1] = next_token

    return X

generated = generate_next_tokens(
    model, tokenizer, test_texts, max_len
)

for text, tokens in zip(test_texts, generated):
    words = [tokenizer.idx2word[t.item()] for t in tokens]
    print(f"Input: {text}")
    print("Generated:", " ".join(words))
    print()



Input: I like
Generated: <CLS> you like bikes <PAD> <PAD>

Input: you
Generated: <CLS> you like bikes <PAD> <PAD>

Input: likes
Generated: <CLS> you like bikes <PAD> <PAD>

